In [7]:
import pandas as pd
import json

# 1. 코퍼스 및 분석 데이터 로드 (텍스트 매핑용 원본)
corpus = pd.read_parquet('../data/insk_corpus.parquet')
analyses = pd.read_parquet('../data/article_analyses.parquet')

# 2. 문서(Document) 텍스트 재조립
merged_df = corpus.merge(analyses, on='article_id', how='left')
# 모델이 문서를 이해할 수 있도록 '제목'과 '요약'을 이어 붙입니다.
merged_df['document_text'] = merged_df['title'].fillna('') + " " + merged_df['summary'].fillna('')

# [핵심 방어 코드] Pandas의 int/float 섞임 현상을 방지하기 위해 무조건 str로 변환
merged_df['article_id_str'] = merged_df['article_id'].astype(str).str.replace(r'\.0$', '', regex=True)

# 기사 ID를 넣으면 조립된 텍스트가 튀어나오는 딕셔너리(사전) 생성
doc_dict = dict(zip(merged_df['article_id_str'], merged_df['document_text']))

# 3. 팀원 A의 Hard Negative 데이터를 활용해 Triplet(질문-정답-오답) 생성
train_triplets = []
jsonl_path = '../data/retrieval_top10_for_reranker.jsonl'

with open(jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        
        # Negative QA(정답 없는 함정 질문)는 파인튜닝에서 제외
        if item.get('type') == 'Negative':
            continue
            
        query = item['question']
        
        # JSON의 ID들도 모두 안전하게 문자열로 변환
        gold_ids = [str(gid) for gid in item.get('gold_articles', [])]
        
        # 문서(md)와 달리 실제 코드는 'hard_negatives' 키를 바로 저장했습니다!
        hard_negative_ids = [str(hn_id) for hn_id in item.get('hard_negatives', [])]
        
        # 번호(ID)를 실제 텍스트로 변환
        positive_texts = [doc_dict[gid] for gid in gold_ids if gid in doc_dict]
        negative_texts = [doc_dict[hn_id] for hn_id in hard_negative_ids if hn_id in doc_dict]
        
        # Reranker 모델이 비교하며 학습할 수 있도록 1:1 쌍으로 엮어줍니다.
        for pos_text in positive_texts:
            for neg_text in negative_texts:
                train_triplets.append({
                    'query': query,
                    'positive': pos_text,
                    'negative': neg_text
                })

print(f"총 {len(train_triplets)}개의 파인튜닝용 Triplet 쌍 완성")

if train_triplets:
    print("\n[데이터 매핑 결과 - 첫 번째 샘플 확인]")
    print(f"Q. (질문) : {train_triplets[0]['query']}")
    print(f"P. (정답) : {train_triplets[0]['positive'][:60]}...")
    print(f"N. (오답) : {train_triplets[0]['negative'][:60]}...")

총 486개의 파인튜닝용 Triplet 쌍 완성

[데이터 매핑 결과 - 첫 번째 샘플 확인]
Q. (질문) : OpenAI가 최근 발표한 신모델·기능은?
P. (정답) : SBS X OpenAI 선거비서, 후보자 공약·정보 '쏙쏙' SBS와 OpenAI가 협력하여 'AI 선거비서...
N. (오답) : [한컴 에이전틱 OS]②PDF 읽는 엔진 오픈소스로, 매출은 애드온으로 한컴이 발표한 '에이전틱 OS' 전략...


In [13]:
# 1. 리스트(train_triplets)를 Pandas DataFrame으로 변환
triplet_df = pd.DataFrame(train_triplets)

# 2. 본문이 너무 길면 표에서 한눈에 보기 어려우므로, 앞 100자만 자른 '미리보기' 컬럼 생성
triplet_df['pos_preview'] = triplet_df['positive'].apply(lambda x: x[:100] + '...' if len(x) > 100 else x)
triplet_df['neg_preview'] = triplet_df['negative'].apply(lambda x: x[:100] + '...' if len(x) > 100 else x)

# 3. 화면에 출력할 컬럼만 깔끔하게 선택 (질문, 정답 미리보기, 오답 미리보기)
preview_df = triplet_df[['query', 'pos_preview', 'neg_preview']]

# 4. 주피터 노트북 설정: 긴 텍스트가 중간에 '...'으로 너무 짧게 잘리지 않도록 길이 조정
pd.set_option('display.max_colwidth', 150)

# 5. 전체 486개 중 무작위로 10개의 샘플을 뽑아서 표(Table) 형태로 출력
display(preview_df.sample(n=7, random_state=29))

,query,pos_preview,neg_preview
281,글로벌 AI 정책·규제 동향은?,"[AI돋보기] ""위험해서 공개 못한다""…스스로 빗장 거는 빅테크 글로벌 빅테크들이 최신 AI 모델의 기능 공개를 제한하며 보안 우려가 커지고 있다. 앤트로픽의 클로드 미토스는 사이...","한글과컴퓨터, AI 돛 달고 글로벌 기업 향해 새출발…사명 '한컴' 변경 한글과컴퓨터(한컴)는 새로운 사명과 AI 중심의 소버린 에이전틱 운영체제(OS) 기업으로의 전환을 발표하며..."
258,AI 스타트업 투자 동향은?,"네카오, '챗GPT+클로드' 멀티 AI 전략…업무 생산성 향상 '집중 투자' 네이버와 카카오는 '챗GPT'와 '클로드'를 도입하여 멀티 AI 전략을 추진하고 있으며, 업무 생산성을...","더벤처스, 온디바이스 AI 전문 스타트업 '아웃오브셋' 시드 투자 더벤처스가 온디바이스 AI 스타트업 아웃오브셋에 시드 투자를 완료했다. 아웃오브셋은 기기에서 직접 구동되는 초경량..."
245,최근 AI M&A 사례는?,"ADI, '엠파워세미컨덕터' 인수…AI 시대 위한 차세대 고집적 전력 포트폴... 아나로그디바이스(ADI)가 엠파워세미컨덕터를 15억 달러에 인수하며 AI 전력 공급 아키텍처를 강...",SoftBank Eyes Switch Inc as It Pushes Deeper Into AI Data Center Expansio... SoftBank Group is explo...
61,Anthropic 매출·사업 동향은?,"앤트로픽, 첫 분기 흑자 눈앞…""AI 산업 첫 '수익 모델' 기대"" 스페이스X가 상장 준비를 하며 AI 분야의 비전을 강조하고, 앤트로픽과 오픈AI도 IPO를 추진하고 있어 AI ...","한화비전, 이집트 시스템 통합 기업 'EMS'와 맞손…중동·아프리카 시장 공략 가속 한화비전이 이집트의 IT 솔루션 기업과 협력하여 북아프리카 시장 확장에 나선다. 이들은 AI 기..."
73,Gemini·구글의 AI 전략은?,"구글 I/O 2026, 검색·개발·쇼핑까지 ‘에이전트 AI’로 재편 2026년 구글 I/O에서 구글은 제미나이 기반의 에이전트 AI 전략을 발표하며 검색, 앱, 개발 도구 등 여러...","마이크로소프트, 앤스로픽에 자체 AI 칩 '마이아' 공급 논의 마이크로소프트가 생성형 AI 스타트업 앤스로픽에 자사 AI 맞춤형 반도체 '마이아' 공급을 논의 중이다. 앤스로픽은 ..."
129,오픈소스 LLM 동향은?,"카카오, LLM 플랫폼 ‘허니비’ 공개… ‘AI 최고위 전략대화’서 소개 카카오는 멀티모달 대규모 언어모델 '허니비'를 오픈소스 형태로 깃허브에 공개하며, 이를 통해 이미지와 텍스...","""GPU 사용부터 LLM 모니터링까지 … AI시대 '옵저버빌리티' 꼭 필요하죠... AI와 클라우드 전환으로 인해 기업들의 IT 인프라 관리가 복잡해지고 있으며, GPU 자원 효율..."
165,NVIDIA AI 칩 동향은?,"코히어, H100 2장으로 구동하는 에이전트용 모델 '커맨드 A+' 오픈 출시 코히어가 기업용 AI 에이전트에 특화된 오픈소스 대형언어모델 '커맨드 A+'를 공개했다. 이 모델은 ...",GPU 시대 다음은 '추론'…Arm 급등에 AI 최적화 기업 주목 AI 인프라 시장에서 Arm Holdings 주가가 최고치를 경신하며 추론 중심으로의 기술 변화가 논의되고 있다....
